In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import AutoMinorLocator

sys.path.append('../..')  # repo root, so `scripts` is importable
from scripts.forecasting_helpers import (
    FEATURES, CATEGORICAL_FEATURES, TARGETS, GROUPING_COLUMNS,
    CLIP_RANGES, PERSISTENCE_MAP,
    load_best_hyperparams, load_and_clean, build_model_inputs, encode_features,
    calculate_all_metrics,
)

np.random.seed(35)

In [2]:
# --- Configuration ---

RCP_FILES = {
    'RCP2.6': '../../data/preprocessed/final_nfi_ch2018_merged/NFI_with_Climate_Averages_RCP26.csv',
    'RCP4.5': '../../data/preprocessed/final_nfi_ch2018_merged/NFI_with_Climate_Averages_RCP45.csv',
    'RCP8.5': '../../data/preprocessed/final_nfi_ch2018_merged/NFI_with_Climate_Averages_RCP85.csv',
}

BEST_HYPERPARAMS_PATH = Path('../../data/hyperparameters/nfi12/best_params.json')

BEST_HYPERPARAMS = load_best_hyperparams(BEST_HYPERPARAMS_PATH)
print(f'Loaded hyperparameters from: {BEST_HYPERPARAMS_PATH}')
for target, params in BEST_HYPERPARAMS.items():
    print(f'  {target}: Lasso alpha={params["Lasso"]["alpha"]:.4f}, '
          f'XGBoost n_estimators={params["XGBoost"]["n_estimators"]}')

Loaded hyperparameters from: ../../data/hyperparameters/nfi12/best_params.json
  BASFPH_next_INVNR: Lasso alpha=0.0500, XGBoost n_estimators=25
  HWSW_prop_next_INVNR: Lasso alpha=0.0050, XGBoost n_estimators=25


In [3]:
def load_and_prepare_data(filepath, features_list, cat_features, targets_list):
    """Load, clean, encode and split into train (INVNR 150/250/350) / test (INVNR 450)."""
    df = load_and_clean(filepath)
    scope = build_model_inputs(df, [150, 250, 350])
    test_df = df[df['INVNR'] == 450].copy()

    X_train = scope['X_train']
    y_train = scope['y_train']
    X_test  = encode_features(test_df, scope['cats_map'], scope['final_features'])
    y_test  = test_df[targets_list].copy()

    return X_train, y_train, X_test, y_test, scope['train_df'], test_df

def train_and_evaluate(X_train, y_train, X_test, y_test,
                       train_df, test_df, hyperparams, grouping_cols):
    """Train Lasso, XGBoost and Persistence baseline; return metrics DataFrame."""
    all_results = []

    for target in y_train.columns:
        models = {
            'Lasso': Pipeline([
                ('scaler', StandardScaler()),
                ('lasso', Lasso(**hyperparams[target]['Lasso'], max_iter=10000))
            ]),
            'XGBoost': xgb.XGBRegressor(
                **hyperparams[target]['XGBoost'],
                importance_type='gain'
            ),
        }

        lo, hi = CLIP_RANGES[target]

        for model_name, model in models.items():
            model.fit(X_train, y_train[target])

            for split_name, X, y_true, orig_df in [
                ('Train', X_train, y_train[target], train_df),
                ('Test',  X_test,  y_test[target],  test_df),
            ]:
                preds = np.clip(model.predict(X), lo, hi)
                eval_df = orig_df.copy()
                eval_df['y_true'] = y_true.values
                eval_df['preds']  = preds

                m = calculate_all_metrics(eval_df['y_true'], eval_df['preds'])
                m.update({'Target': target, 'Model': model_name, 'Split': split_name})
                all_results.append(m)

                for gc in grouping_cols:
                    if gc not in eval_df.columns:
                        continue
                    for gv in sorted(eval_df[gc].unique()):
                        gdf = eval_df[eval_df[gc] == gv]
                        gm = calculate_all_metrics(gdf['y_true'], gdf['preds'],
                                                   group_name=f'{gc}_{gv}')
                        gm.update({'Target': target, 'Model': model_name, 'Split': split_name})
                        all_results.append(gm)

        # Persistence baseline
        cur_col = PERSISTENCE_MAP[target]
        for split_name, y_true, orig_df in [
            ('Train', y_train[target], train_df),
            ('Test',  y_test[target],  test_df),
        ]:
            preds = np.clip(orig_df[cur_col].values, lo, hi)
            eval_df = orig_df.copy()
            eval_df['y_true'] = y_true.values
            eval_df['preds']  = preds

            m = calculate_all_metrics(eval_df['y_true'], eval_df['preds'])
            m.update({'Target': target, 'Model': 'Persistence', 'Split': split_name})
            all_results.append(m)

            for gc in grouping_cols:
                if gc not in eval_df.columns:
                    continue
                for gv in sorted(eval_df[gc].unique()):
                    gdf = eval_df[eval_df[gc] == gv]
                    gm = calculate_all_metrics(gdf['y_true'], gdf['preds'],
                                               group_name=f'{gc}_{gv}')
                    gm.update({'Target': target, 'Model': 'Persistence', 'Split': split_name})
                    all_results.append(gm)

    return pd.DataFrame(all_results)

In [4]:
# --- Run evaluation for every RCP scenario ---

rcp_results = {}

for rcp_label, filepath in RCP_FILES.items():
    print(f'\n{"="*60}')
    print(f'  {rcp_label}')
    print(f'{"="*60}')

    X_train, y_train, X_test, y_test, train_df, test_df = load_and_prepare_data(
        filepath, FEATURES, CATEGORICAL_FEATURES, TARGETS
    )
    print(f'  Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')

    results_df = train_and_evaluate(
        X_train, y_train, X_test, y_test,
        train_df, test_df, BEST_HYPERPARAMS, GROUPING_COLUMNS
    )
    results_df['RCP'] = rcp_label
    rcp_results[rcp_label] = results_df
    print(f'  Done.')

all_results = pd.concat(rcp_results.values(), ignore_index=True)
print(f'\nTotal metric rows collected: {len(all_results)}')


  RCP2.6
  Train: 360 rows  |  Test: 120 rows
  Done.

  RCP4.5
  Train: 360 rows  |  Test: 120 rows
  Done.

  RCP8.5
  Train: 360 rows  |  Test: 120 rows
  Done.

Total metric rows collected: 504


In [5]:
# --- Overall test-set metrics table ---

METRIC_COLS = ['N', 'RMSE', '%RMSE', 'R_squared', 'Bias', '%Bias']

overall_test = (
    all_results[
        (all_results['Split'] == 'Test') &
        (all_results['Group'] == 'Overall')
    ]
    .copy()
    .sort_values(['Target', 'Model', 'RCP'])
)

print('Overall test-set metrics across RCP scenarios')
print('=' * 70)
for target in overall_test['Target'].unique():
    for model in overall_test['Model'].unique():
        subset = overall_test[
            (overall_test['Target'] == target) &
            (overall_test['Model'] == model)
        ]
        if subset.empty:
            continue
        print(f'\n{target}  |  {model}')
        display(
            subset.set_index('RCP')[METRIC_COLS]
                  .style.format({
                      'RMSE': '{:.3f}', '%RMSE': '{:.1f}',
                      'R_squared': '{:.3f}', 'Bias': '{:.3f}', '%Bias': '{:.1f}'
                  })
        )

Overall test-set metrics across RCP scenarios

BASFPH_next_INVNR  |  Lasso


,N,RMSE,%RMSE,R_squared,Bias,%Bias
RCP,,,,,,
RCP2.6,120,3.678,10.5,0.893,-0.357,-1.0
RCP4.5,120,3.694,10.5,0.892,-0.475,-1.4
RCP8.5,120,3.676,10.5,0.893,-0.351,-1.0



BASFPH_next_INVNR  |  Persistence


,N,RMSE,%RMSE,R_squared,Bias,%Bias
RCP,,,,,,
RCP2.6,120,3.912,11.2,0.879,-1.372,-3.9
RCP4.5,120,3.912,11.2,0.879,-1.372,-3.9
RCP8.5,120,3.912,11.2,0.879,-1.372,-3.9



BASFPH_next_INVNR  |  XGBoost


...

,N,RMSE,%RMSE,R_squared,Bias,%Bias
RCP,,,,,,
RCP2.6,120,0.043,7.7,0.967,-0.000,-0.0
RCP4.5,120,0.043,7.7,0.967,-0.000,-0.0
RCP8.5,120,0.043,7.7,0.967,-0.000,-0.0



HWSW_prop_next_INVNR  |  Persistence


,N,RMSE,%RMSE,R_squared,Bias,%Bias
RCP,,,,,,
RCP2.6,120,0.045,8.1,0.964,-0.015,-2.7
RCP4.5,120,0.045,8.1,0.964,-0.015,-2.7
RCP8.5,120,0.045,8.1,0.964,-0.015,-2.7



HWSW_prop_next_INVNR  |  XGBoost


,N,RMSE,%RMSE,R_squared,Bias,%Bias
RCP,,,,,,
RCP2.6,120,0.062,11.1,0.932,-0.010,-1.8
RCP4.5,120,0.063,11.2,0.930,-0.010,-1.8
RCP8.5,120,0.062,11.0,0.932,-0.009,-1.7
